# LeetCode 4003 — Minimum Cost Path with Alternating Directions III bằng Python

- Link: https://leetcode.com/problems/minimum-cost-path-with-alternating-directions-iii/
- Độ khó: **Hard**
- Chủ đề: Đồ thị, Đường đi ngắn nhất (Dijkstra), Hàng đợi ưu tiên (Heap), Ma trận, Nén trạng thái theo tính chẵn lẻ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minCost(self, m: int, n: int, penalty: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Bạn đứng ở ô `(0, 0)` của lưới `m × n` và muốn tới ô `(m-1, n-1)`. Mỗi lần **vào** ô `(i, j)` tốn `(i+1)(j+1)`. Các hành động được đánh số `1, 2, 3, ...`: hành động **lẻ** nên đi **phải/xuống**, hành động **chẵn** nên đi **trái/lên**. Đi sai luật thì trả thêm `penalty` của **ô xuất phát**. Bạn cũng có thể **đứng yên** (chờ) và trả `penalty` của ô đang đứng — và hành động kế tiếp vẫn đổi tính chẵn lẻ.

Bài trông như một bài đường đi ngắn nhất thông thường, nhưng luật "chẵn/lẻ" khiến **chi phí của một bước phụ thuộc vào số bước đã đi**. Mấu chốt: ta không cần biết chính xác số bước, chỉ cần biết nó **chẵn hay lẻ**. Gắn thêm 1 bit chẵn/lẻ vào trạng thái là xong: đồ thị `2·m·n` đỉnh, mọi cạnh có chi phí không âm → **Dijkstra**.

Sau notebook này bạn sẽ:

1. Đọc kỹ luật đề (nhập ô, chờ, phạt) và tính tay các ví dụ.
2. Nén "thời gian" thành **một bit tính chẵn lẻ** và dựng đồ thị trạng thái `(i, j, parity)`.
3. Viết brute force vét cạn mọi dãy hành động và DP theo số hành động (Bellman-Ford theo lớp thời gian) để kiểm chứng.
4. Viết Dijkstra với `heapq`, và SPFA để thấy vì sao Dijkstra tốt hơn.
5. Đối chiếu các cách bằng test ngẫu nhiên có seed; hiểu độ phức tạp `O(mn log(mn))`.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Lưới `m` hàng, `n` cột. Bắt đầu ở `(0,0)`, **trả ngay** chi phí vào ô đó `1·1 = 1`.
- Mỗi **hành động** (đánh số từ 1) là một trong: đi sang ô kề (4 hướng) hoặc đứng yên.
- Hành động **lẻ**: hướng hợp lệ là **phải** hoặc **xuống**. Hành động **chẵn**: hướng hợp lệ là **trái** hoặc **lên**.
- Chi phí một hành động:
  - Đi **đúng luật**: chỉ trả chi phí vào ô đích `(x+1)(y+1)`.
  - Đi **sai luật**: trả chi phí vào ô đích **cộng** `penalty[i][j]` với `(i, j)` là ô **xuất phát**.
  - **Đứng yên** tại `(i, j)`: trả `penalty[i][j]`.
- Sau mỗi hành động (kể cả có phạt hay đứng yên), số thứ tự hành động tăng 1 nên luật chẵn/lẻ **luôn đổi**.
- Trả về tổng chi phí nhỏ nhất để tới `(m-1, n-1)`.

**Ví dụ 1:** `m=2, n=2, penalty=[[5,3],[1,4]]` → `8`: `1` (vào `(0,0)`) + `2` (xuống `(1,0)`, đúng luật lẻ) + `4 + 1` (sang phải `(1,1)` vào hành động chẵn: sai luật, phạt `penalty[1][0] = 1`).

**Ví dụ 2:** `m=2, n=2, penalty=[[0,7],[3,2]]` → `7`: `1` + chờ ở `(0,0)` (phạt `0`) + sang phải `(0,1)` ở hành động chẵn (sai luật, phạt `penalty[0][0] = 0`, vào `2`) + xuống `(1,1)` ở hành động lẻ (đúng luật, vào `4`): `1+0+2+0+4 = 7`.

**Ví dụ 3:** `m=2, n=3, penalty=[[8,0,9],[7,4,1]]` → `12`.

**Ràng buộc**

- `1 <= m, n <= 10^5`, `2 <= m*n <= 10^5`
- `penalty.length == m`, `penalty[i].length == n`
- `0 <= penalty[i][j] <= 10^5`

Lưu ý: `m * n ≤ 10^5` nên lưới có thể là `316 × 316`, hoặc rất dài như `1 × 100000`.

In [1]:
from typing import List

EXAMPLES = [
    (2, 2, [[5, 3], [1, 4]], 8),
    (2, 2, [[0, 7], [3, 2]], 7),
    (2, 3, [[8, 0, 9], [7, 4, 1]], 12),
]
for m, n, pen, expected in EXAMPLES:
    print(f"m={m}, n={n}, penalty={pen} -> kỳ vọng {expected}")

m=2, n=2, penalty=[[5, 3], [1, 4]] -> kỳ vọng 8
m=2, n=2, penalty=[[0, 7], [3, 2]] -> kỳ vọng 7
m=2, n=3, penalty=[[8, 0, 9], [7, 4, 1]] -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Toạ độ ô, 4 hướng và kiểm tra biên

Ta di chuyển bằng cặp `(di, dj)`: lên `(-1, 0)`, phải `(0, 1)`, trái `(0, -1)`, xuống `(1, 0)`. Trước khi dùng ô mới `(x, y)` phải kiểm tra `0 <= x < m` và `0 <= y < n`.

In [2]:
m, n = 2, 3
DIRS = [(-1, 0), (0, 1), (0, -1), (1, 0)]     # lên, phải, trái, xuống
i, j = 0, 0
for di, dj in DIRS:
    x, y = i + di, j + dj
    print((di, dj), "->", (x, y), "hợp lệ" if 0 <= x < m and 0 <= y < n else "ra ngoài lưới")

(-1, 0) -> (-1, 0) ra ngoài lưới
(0, 1) -> (0, 1) hợp lệ
(0, -1) -> (0, -1) ra ngoài lưới
(1, 0) -> (1, 0) hợp lệ


### 3.2. Hàng đợi ưu tiên `heapq`

`heapq` biến một list thành **min-heap**: `heappush(h, x)` thêm, `heappop(h)` lấy phần tử **nhỏ nhất**, đều `O(log N)`. Với tuple, so sánh theo thứ tự từ điển, nên đặt **chi phí ở đầu tuple** để chi phí nhỏ nhất ra trước.

In [3]:
import heapq
h = []
for item in [(7, "c"), (2, "a"), (5, "b"), (2, "z")]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(len(h))])    # theo chi phí tăng dần

[(2, 'a'), (2, 'z'), (5, 'b'), (7, 'c')]


### 3.3. Dijkstra trong 5 dòng, và "lazy deletion"

**Dijkstra** tìm đường ngắn nhất từ một nguồn khi mọi cạnh có chi phí **không âm**. Giữ `dist[v]` = chi phí tốt nhất đã biết. Mỗi lần lấy từ heap trạng thái có `d` nhỏ nhất; nếu `d > dist[v]` thì bản ghi đã cũ (lazy deletion) → bỏ qua. Ngược lại, thử cải thiện hàng xóm: `dist[w] = d + chi_phí_cạnh`. Lần đầu **lấy ra** (không phải lần đầu đẩy vào) đích `t` chính là khoảng cách ngắn nhất tới `t`.

In [4]:
def dijkstra_demo(adj, src, dst):
    dist = {src: 0}
    pq = [(0, src)]
    while pq:
        d, v = heapq.heappop(pq)
        if v == dst:
            return d
        if d > dist.get(v, float("inf")):
            continue                              # bản ghi cũ
        for w, c in adj.get(v, []):
            nd = d + c
            if nd < dist.get(w, float("inf")):
                dist[w] = nd
                heapq.heappush(pq, (nd, w))
    return -1

adj = {"A": [("B", 5), ("C", 1)], "C": [("B", 2)], "B": [("D", 1)]}
print(dijkstra_demo(adj, "A", "D"))               # A->C->B->D = 4

4


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `penalty = [[0,7],[3,2]]`.

- Bắt đầu: tốn `1`.
- Cách "đi thẳng": hành động 1 (lẻ) sang phải `(0,1)` tốn `2`; hành động 2 (chẵn) đi xuống `(1,1)` là **sai luật**: `4 + penalty[0][1] = 4 + 7 = 11`. Tổng `1 + 2 + 11 = 14`.
- Cách "chờ trước": hành động 1 chờ ở `(0,0)` tốn `penalty[0][0] = 0`. Giờ là hành động 2 (chẵn): sang phải `(0,1)` sai luật: `2 + penalty[0][0] = 2`. Hành động 3 (lẻ): xuống `(1,1)` đúng luật, `4`. Tổng `1 + 0 + 2 + 4 = 7`.

Bài học: **chờ có thể đáng tiền** nếu `penalty` ô hiện tại nhỏ, vì nó "đổi pha" chẵn/lẻ cho bước sau. Vậy khi ta đứng ở một ô, chi phí các bước tiếp theo **phụ thuộc vào việc hành động kế tiếp là chẵn hay lẻ**. Tại mỗi ô ta cần nhớ thêm "hành động kế tiếp là lẻ hay chẵn".

Ta viết hàm tính chi phí một hành động theo đúng luật đề để kiểm tra bằng tay.

In [5]:
def action_cost(m, n, penalty, i, j, di, dj, t):
    """Chi phí hành động số t (t >= 1) từ ô (i,j) theo hướng (di,dj); (0,0) = đứng yên.
    Trả về (chi_phí, ô_mới) hoặc None nếu đi ra ngoài lưới."""
    p = penalty[i][j]
    if (di, dj) == (0, 0):
        return p, (i, j)
    x, y = i + di, j + dj
    if not (0 <= x < m and 0 <= y < n):
        return None
    right_or_down = (di, dj) in ((0, 1), (1, 0))
    allowed = right_or_down if t % 2 == 1 else not right_or_down
    return (x + 1) * (y + 1) + (0 if allowed else p), (x, y)

pen = [[0, 7], [3, 2]]
# cách đi thẳng: t=1 phải, t=2 xuống (sai luật)
c1, cell = action_cost(2, 2, pen, 0, 0, 0, 1, 1)
c2, cell = action_cost(2, 2, pen, 0, 1, 1, 0, 2)
print("đi thẳng:", 1 + c1 + c2)
# cách chờ trước: t=1 chờ, t=2 phải (sai), t=3 xuống (đúng)
a, _ = action_cost(2, 2, pen, 0, 0, 0, 0, 1)
b, _ = action_cost(2, 2, pen, 0, 0, 0, 1, 2)
c, _ = action_cost(2, 2, pen, 0, 1, 1, 0, 3)
print("chờ trước:", 1 + a + b + c)

đi thẳng: 14
chờ trước: 7


## 5. Cách 1 — Brute Force: thử mọi dãy hành động

**Ý tưởng.** Mỗi hành động có tối đa 5 lựa chọn (4 hướng + đứng yên). Ta đệ quy dãy hành động với độ sâu tối đa `L`, mỗi lần tới đích thì cập nhật đáp án (có cắt tỉa: bỏ nhánh có chi phí ≥ đáp án tốt nhất). Vì chi phí không âm nên đường tốt nhất không cần dài; nhưng ta chưa biết cần `L` bao nhiêu, nên chỉ dùng cho lưới rất nhỏ với `L` thoải mái (`L = 2·m·n + 2`, ta sẽ giải thích vì sao ở mục 10).

Đây là bản dịch gần như nguyên văn của đề nên rất đáng tin để làm trọng tài.

In [6]:
DIRS5 = [(0, 0), (-1, 0), (0, 1), (0, -1), (1, 0)]

def brute_force(m, n, penalty, L=None):
    if L is None:
        L = 2 * m * n + 2
    best = [float("inf")]
    def dfs(i, j, t, cost):
        # t = số hành động đã làm; hành động tiếp theo là t + 1
        if cost >= best[0]:
            return
        if (i, j) == (m - 1, n - 1):
            best[0] = cost
        if t == L:
            return
        for di, dj in DIRS5:
            res = action_cost(m, n, penalty, i, j, di, dj, t + 1)
            if res is None:
                continue
            c, (x, y) = res
            dfs(x, y, t + 1, cost + c)
    dfs(0, 0, 0, 1)
    return best[0]

for m, n, pen, expected in EXAMPLES[:2]:
    print(m, n, pen, "->", brute_force(m, n, pen), "(kỳ vọng", expected, ")")

2 2 [[5, 3], [1, 4]] -> 8 (kỳ vọng 8 )
2 2 [[0, 7], [3, 2]] -> 7 (kỳ vọng 7 )


**Độ phức tạp.** `O(5^L)` — bùng nổ. Với `2×3` (`L = 14`) đã rất chậm, nên ta chỉ chạy vét cạn trên lưới tối đa 4 ô (có cắt tỉa theo chi phí). **Nút thắt:** cùng một "tình huống" (đang ở ô nào, hành động kế là lẻ hay chẵn) được lặp lại vô số lần với các chi phí tích luỹ khác nhau; ta chỉ cần giữ chi phí **nhỏ nhất**.

## 6. Bước đột phá — trạng thái `(i, j, parity)`

**Quan sát 1.** Luật chẵn/lẻ chỉ dựa vào số thứ tự hành động **modulo 2**. Gọi `k = 1` nếu hành động kế tiếp là lẻ, `k = 0` nếu chẵn. Sau mỗi hành động `k` đổi (`k ^= 1`). Do đó mọi thông tin cần thiết về tương lai là **ô hiện tại và `k`**.

**Quan sát 2.** Mỗi hành động từ trạng thái `(i, j, k)` có chi phí **chỉ phụ thuộc** `(i, j, k)` và lựa chọn:

- Đứng yên: tới `(i, j, k^1)`, chi phí `penalty[i][j]`.
- Đi tới `(x, y)`: tới `(x, y, k^1)`, chi phí `(x+1)(y+1)` cộng `penalty[i][j]` nếu hướng **không** khớp `k`.

**Quan sát 3.** Mọi chi phí `≥ 0`. Vậy đây là bài **đường đi ngắn nhất** trên đồ thị `2·m·n` đỉnh, mỗi đỉnh có `≤ 5` cạnh. Điểm bắt đầu: `(0, 0, k=1)` với chi phí ban đầu `1`. Đích: `(m-1, n-1, k)` với `k` bất kỳ → lấy cái nào được lấy ra khỏi heap trước.

**Quy tắc "nếu … thì …"** cho một cạnh đi:

- nếu hướng là phải/xuống và `k = 1` → đúng luật, không phạt;
- nếu hướng là trái/lên và `k = 0` → đúng luật, không phạt;
- còn lại → phạt `penalty` của ô xuất phát.

Ta kiểm chứng ý tưởng bằng cách dựng đồ thị trạng thái và đếm đỉnh/cạnh.

In [7]:
def state_edges(m, n, penalty, i, j, k):
    """Liệt kê các cạnh ra từ trạng thái (i, j, k): danh sách (chi_phí, i', j', k')."""
    out = []
    p = penalty[i][j]
    out.append((p, i, j, k ^ 1))                         # đứng yên
    for di, dj in DIRS:                                  # lên, phải, trái, xuống
        x, y = i + di, j + dj
        if 0 <= x < m and 0 <= y < n:
            matches = ((di, dj) in ((0, 1), (1, 0))) == (k == 1)
            out.append(((x + 1) * (y + 1) + (0 if matches else p), x, y, k ^ 1))
    return out

pen = [[5, 3], [1, 4]]
for k in (1, 0):
    print(f"từ (0,0) với k={k}:", state_edges(2, 2, pen, 0, 0, k))
total_edges = sum(len(state_edges(2, 2, pen, i, j, k)) for i in range(2) for j in range(2) for k in (0, 1))
print("số đỉnh:", 2 * 2 * 2, " số cạnh:", total_edges)

từ (0,0) với k=1: [(5, 0, 0, 0), (2, 0, 1, 0), (2, 1, 0, 0)]
từ (0,0) với k=0: [(5, 0, 0, 1), (7, 0, 1, 1), (7, 1, 0, 1)]
số đỉnh: 8  số cạnh: 24


## 7. Các cách tiếp theo

- **Cách 2:** DP theo **số hành động** (Bellman-Ford theo lớp thời gian) — dùng làm trọng tài thứ hai cho lưới vừa.
- **Cách 3:** **Dijkstra** trên `(i, j, parity)` — đáp án nộp.
- **Cách 4:** **SPFA** (Bellman-Ford có hàng đợi) trên cùng đồ thị — đúng nhưng xấu nhất chậm hơn.

### 7.1. Cách 2 — DP theo số hành động

**Ý tưởng.** Gọi `cur[i][j]` = chi phí nhỏ nhất để đang ở `(i, j)` sau đúng `t` hành động. Parity của hành động kế tiếp **được suy ra từ `t`** nên không cần thêm chiều. Mỗi lớp `t → t+1` thử 5 lựa chọn từ mọi ô. Đáp án = `min` theo `t` của `cur[m-1][n-1]`. Cần chọn `T` đủ lớn: một đường đi ngắn nhất trong đồ thị trạng thái có ≤ `2mn` đỉnh nên `T = 2mn` hành động là đủ (chứng minh ở mục 10).

In [8]:
def layered_dp(m, n, penalty, T=None):
    if T is None:
        T = 2 * m * n + 2
    INF = float("inf")
    cur = [[INF] * n for _ in range(m)]
    cur[0][0] = 1
    best = 1 if (m - 1, n - 1) == (0, 0) else INF
    for t in range(1, T + 1):                          # hành động số t
        nxt = [[INF] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                c = cur[i][j]
                if c == INF:
                    continue
                for di, dj in DIRS5:
                    res = action_cost(m, n, penalty, i, j, di, dj, t)
                    if res is None:
                        continue
                    cost, (x, y) = res
                    if c + cost < nxt[x][y]:
                        nxt[x][y] = c + cost
        cur = nxt
        best = min(best, cur[m - 1][n - 1])
    return best

for m, n, pen, expected in EXAMPLES:
    assert layered_dp(m, n, pen) == expected
print("Cách 2 khớp 3 ví dụ")

Cách 2 khớp 3 ví dụ


**Độ phức tạp.** `O(T · m · n · 5) = O((mn)²)`. Với `mn = 10^5` là quá chậm; nhưng đủ nhanh cho `mn ≤ 40` để kiểm chứng.

### 7.2. Cách 3 — Dijkstra trên `(i, j, parity)` (đáp án nộp)

**Giải thích từng đoạn.**

1. `dist[i][j][k]`: chi phí nhỏ nhất để ở `(i, j)` khi hành động kế có parity `k`. Khởi tạo vô cực, trừ `dist[0][0][1] = 1` (đã trả phí vào `(0,0)`, hành động 1 là lẻ).
2. Heap chứa `(chi phí, i, j, k)`; lấy phần tử nhỏ nhất. Nếu `(i, j)` là đích → trả về (lần đầu lấy ra là tối ưu vì chi phí không âm).
3. Bản ghi cũ `d > dist[i][j][k]` → bỏ qua.
4. Cạnh **đứng yên**: thêm `penalty[i][j]`, đổi `k`.
5. Cạnh **đi**: với 4 hướng, tính chi phí vào ô đích, cộng phạt nếu hướng không khớp `k`, đổi `k`.

In [9]:
from heapq import heappush, heappop

def dijkstra_solve(m, n, penalty):
    INF = float("inf")
    dist = [[[INF, INF] for _ in range(n)] for _ in range(m)]
    dist[0][0][1] = 1
    pq = [(1, 0, 0, 1)]
    while pq:
        d, i, j, k = heappop(pq)
        if i == m - 1 and j == n - 1:
            return d
        if d > dist[i][j][k]:
            continue
        p = penalty[i][j]
        # đứng yên
        if d + p < dist[i][j][k ^ 1]:
            dist[i][j][k ^ 1] = d + p
            heappush(pq, (d + p, i, j, k ^ 1))
        # đi 4 hướng
        for di, dj in DIRS:
            x, y = i + di, j + dj
            if 0 <= x < m and 0 <= y < n:
                matches = ((di, dj) in ((0, 1), (1, 0))) == (k == 1)
                nd = d + (x + 1) * (y + 1) + (0 if matches else p)
                if nd < dist[x][y][k ^ 1]:
                    dist[x][y][k ^ 1] = nd
                    heappush(pq, (nd, x, y, k ^ 1))
    return -1

for m, n, pen, expected in EXAMPLES:
    assert dijkstra_solve(m, n, pen) == expected
print("Cách 3 khớp 3 ví dụ")

Cách 3 khớp 3 ví dụ


**Độ phức tạp.** `V = 2mn` đỉnh, `E ≤ 5V` cạnh: `O(E log V) = O(mn log(mn))` thời gian, `O(mn)` bộ nhớ.

### 7.3. Cách 4 — SPFA (Bellman-Ford có hàng đợi)

**Ý tưởng.** Không cần heap: dùng hàng đợi thường; mỗi khi `dist[v]` giảm ta đưa `v` vào hàng đợi nếu chưa có. Đúng với mọi đồ thị không có chu trình âm, nhưng **xấu nhất** có thể quét một đỉnh nhiều lần (`O(V·E)`). Ở đây chi phí không âm nên Dijkstra hiệu quả hơn hẳn; SPFA hữu ích để minh hoạ và kiểm chứng chéo.

In [10]:
from collections import deque

def spfa_solve(m, n, penalty):
    INF = float("inf")
    dist = [[[INF, INF] for _ in range(n)] for _ in range(m)]
    dist[0][0][1] = 1
    inq = [[[False, False] for _ in range(n)] for _ in range(m)]
    q = deque([(0, 0, 1)])
    inq[0][0][1] = True
    while q:
        i, j, k = q.popleft()
        inq[i][j][k] = False
        d = dist[i][j][k]
        for cost, x, y, nk in state_edges(m, n, penalty, i, j, k):
            if d + cost < dist[x][y][nk]:
                dist[x][y][nk] = d + cost
                if not inq[x][y][nk]:
                    inq[x][y][nk] = True
                    q.append((x, y, nk))
    return min(dist[m - 1][n - 1])

for m, n, pen, expected in EXAMPLES:
    assert spfa_solve(m, n, pen) == expected
print("Cách 4 khớp 3 ví dụ")

Cách 4 khớp 3 ví dụ


**Độ phức tạp.** `O(V · E)` xấu nhất, trung bình thường gần `O(E)`. Dùng khi có cạnh âm (không có ở bài này).

## 8. TRACE từng bước

Ta in từng lần lấy ra từ heap của Dijkstra trên ví dụ 1, kèm các cạnh được nới lỏng thành công. Ký hiệu `k=1` là hành động kế tiếp **lẻ** (nên đi phải/xuống), `k=0` là **chẵn** (nên đi trái/lên).

| cột | ý nghĩa |
|---|---|
| `pop` | trạng thái và chi phí vừa lấy khỏi heap |
| `thử` | `đứng yên` hoặc hướng đi, chi phí cộng thêm, trạng thái mới |

In [11]:
def trace_dijkstra(m, n, penalty):
    INF = float("inf")
    names = {(-1, 0): "lên", (0, 1): "phải", (0, -1): "trái", (1, 0): "xuống"}
    dist = [[[INF, INF] for _ in range(n)] for _ in range(m)]
    dist[0][0][1] = 1
    pq = [(1, 0, 0, 1)]
    step = 0
    while pq:
        d, i, j, k = heappop(pq)
        if d > dist[i][j][k]:
            continue
        step += 1
        print(f"pop #{step}: ô ({i},{j}) k={k} chi phí={d}")
        if i == m - 1 and j == n - 1:
            print("=> tới đích, đáp án", d)
            return d
        p = penalty[i][j]
        if d + p < dist[i][j][k ^ 1]:
            dist[i][j][k ^ 1] = d + p
            heappush(pq, (d + p, i, j, k ^ 1))
            print(f"   thử đứng yên (+{p}) -> ({i},{j}) k={k ^ 1}: {d + p}")
        for di, dj in DIRS:
            x, y = i + di, j + dj
            if 0 <= x < m and 0 <= y < n:
                matches = ((di, dj) in ((0, 1), (1, 0))) == (k == 1)
                add = (x + 1) * (y + 1) + (0 if matches else p)
                if d + add < dist[x][y][k ^ 1]:
                    dist[x][y][k ^ 1] = d + add
                    heappush(pq, (d + add, x, y, k ^ 1))
                    print(f"   thử {names[(di, dj)]} (+{add}{'' if matches else ', gồm phạt ' + str(p)}) -> ({x},{y}) k={k ^ 1}: {d + add}")
    return -1

trace_dijkstra(2, 2, [[5, 3], [1, 4]])

pop #1: ô (0,0) k=1 chi phí=1
   thử đứng yên (+5) -> (0,0) k=0: 6
   thử phải (+2) -> (0,1) k=0: 3
   thử xuống (+2) -> (1,0) k=0: 3
pop #2: ô (0,1) k=0 chi phí=3
   thử đứng yên (+3) -> (0,1) k=1: 6
   thử xuống (+7, gồm phạt 3) -> (1,1) k=1: 10
pop #3: ô (1,0) k=0 chi phí=3
   thử đứng yên (+1) -> (1,0) k=1: 4
   thử phải (+5, gồm phạt 1) -> (1,1) k=1: 8
pop #4: ô (1,0) k=1 chi phí=4
   thử phải (+4) -> (1,1) k=0: 8
pop #5: ô (0,0) k=0 chi phí=6
pop #6: ô (0,1) k=1 chi phí=6
pop #7: ô (1,1) k=0 chi phí=8
=> tới đích, đáp án 8


8

Đọc trace: từ `(0,0)` ta lấy chi phí `1` và thử sang phải/xuống (đúng luật lẻ, chi phí vào ô), hoặc đứng yên (phạt `5`, giờ `k=0`). Đường tối ưu `(0,0) → (1,0) → (1,1)` được xác định khi `(1,1)` bị lấy ra với chi phí `8`. Ta trace thêm ví dụ 2 (có "chờ"):

In [12]:
trace_dijkstra(2, 2, [[0, 7], [3, 2]])

pop #1: ô (0,0) k=1 chi phí=1
   thử đứng yên (+0) -> (0,0) k=0: 1
   thử phải (+2) -> (0,1) k=0: 3
   thử xuống (+2) -> (1,0) k=0: 3
pop #2: ô (0,0) k=0 chi phí=1
   thử phải (+2, gồm phạt 0) -> (0,1) k=1: 3
   thử xuống (+2, gồm phạt 0) -> (1,0) k=1: 3
pop #3: ô (0,1) k=0 chi phí=3
   thử xuống (+11, gồm phạt 7) -> (1,1) k=1: 14
pop #4: ô (0,1) k=1 chi phí=3
   thử xuống (+4) -> (1,1) k=0: 7
pop #5: ô (1,0) k=0 chi phí=3
   thử phải (+7, gồm phạt 3) -> (1,1) k=1: 10
pop #6: ô (1,0) k=1 chi phí=3
pop #7: ô (1,1) k=0 chi phí=7
=> tới đích, đáp án 7


7

## 9. Code nộp LeetCode

In [13]:
from typing import List
from heapq import heappush, heappop
from math import inf

class Solution:
    def minCost(self, m: int, n: int, penalty: List[List[int]]) -> int:
        dist = [[[inf, inf] for _ in range(n)] for _ in range(m)]
        dist[0][0][1] = 1                       # đã trả phí vào (0,0); hành động kế (số 1) là lẻ
        pq = [(1, 0, 0, 1)]
        dirs = ((-1, 0), (0, 1), (0, -1), (1, 0))   # lên, phải, trái, xuống
        while pq:
            d, i, j, k = heappop(pq)
            if i == m - 1 and j == n - 1:
                return d
            if d > dist[i][j][k]:
                continue
            p = penalty[i][j]
            nd = d + p                          # đứng yên: trả phạt, đổi parity
            if nd < dist[i][j][k ^ 1]:
                dist[i][j][k ^ 1] = nd
                heappush(pq, (nd, i, j, k ^ 1))
            for idx, (dx, dy) in enumerate(dirs):
                x, y = i + dx, j + dy
                if 0 <= x < m and 0 <= y < n:
                    # idx & 1 == 1 với phải/xuống: khớp luật khi k == 1; trái/lên khớp khi k == 0
                    nd = d + (x + 1) * (y + 1) + ((idx & 1) ^ k) * p
                    if nd < dist[x][y][k ^ 1]:
                        dist[x][y][k ^ 1] = nd
                        heappush(pq, (nd, x, y, k ^ 1))
        return -1

for m, n, pen, expected in EXAMPLES:
    assert Solution().minCost(m, n, pen) == expected
print("Code nộp khớp 3 ví dụ")

Code nộp khớp 3 ví dụ


Mẹo bit: trong `dirs`, các hướng ở chỉ số lẻ (`1` = phải, `3` = xuống) là hướng "phải/xuống", nên `idx & 1` là `1` cho phải/xuống và `0` cho trái/lên. Khi `k = 1` (hành động lẻ), hướng khớp luật nếu `idx & 1 == 1`, tức `(idx & 1) ^ k == 0`; khi `k = 0` khớp nếu `idx & 1 == 0`, cũng `(idx & 1) ^ k == 0`. Vì vậy `((idx & 1) ^ k) * p` là phạt cần cộng: `0` nếu khớp, `p` nếu sai luật.

## 10. Vì sao thuật toán đúng

**(a) Trạng thái đầy đủ.** Chi phí và tính hợp lệ của một hành động phụ thuộc: ô hiện tại, hướng, và **parity của số thứ tự hành động**. Parity đổi sau **mọi** hành động (đi, đi sai luật, đứng yên) đúng như đề. Vậy `(i, j, k)` quyết định toàn bộ tương lai, không cần nhớ lịch sử.

**(b) Chi phí không âm.** `(x+1)(y+1) ≥ 1`, `penalty ≥ 0`. Dijkstra đúng khi mọi cạnh không âm: khi một trạng thái được lấy khỏi heap với chi phí `d`, không còn đường nào rẻ hơn tới nó (mọi đường khác phải đi qua một trạng thái đang trong heap có chi phí `≥ d`, và cộng thêm các cạnh không âm).

**(c) Điểm kết thúc.** Mục tiêu là tới ô `(m-1, n-1)` với parity bất kỳ. Trạng thái đầu tiên có `(i, j) = (m-1, n-1)` lấy ra khỏi heap có chi phí nhỏ nhất trong hai parity, đúng đáp án.

**(d) Giới hạn độ dài đường ngắn nhất (cho brute force).** Trong đồ thị trạng thái có `2mn` đỉnh, một đường đi ngắn nhất không lặp đỉnh nào (nếu lặp thì cắt chu trình đi, chi phí không tăng vì không âm). Vậy tối đa `2mn` đỉnh → `≤ 2mn − 1` hành động. Do đó `T = 2mn + 2` hành động là đủ để brute force và DP theo lớp không bỏ sót.

**(e) Luôn có đường.** Từ `(0,0)` với `m, n ≥ 1`, `mn ≥ 2`: luôn tới được đích bằng cách đi phải/xuống ở hành động lẻ, và ở hành động chẵn thì đứng yên hoặc đi sai luật. Nên đáp án luôn hữu hạn; `return -1` cuối hàm chỉ là phòng thủ.

## 11. Phân tích Time/Space

Gọi `N = m · n ≤ 10^5` là số ô.

| Cách | Time | Space |
|---|---|---|
| Brute force | `O(5^L)`, `L = 2N + 2` | `O(L)` |
| DP theo lớp | `O(N · T) = O(N²)` | `O(N)` |
| Dijkstra (nộp) | `O(N log N)` | `O(N)` |
| SPFA | `O(N · E)` xấu nhất | `O(N)` |

Dijkstra có `2N` đỉnh và `≤ 5·2N` cạnh; mỗi cạnh có thể đẩy một bản ghi vào heap → `O(N log N)`. Ta đo thực tế trên các hình dạng lưới khác nhau (cùng `N = 10^5`):

In [14]:
import time, random

def timed(m, n, pen):
    t0 = time.time()
    r = Solution().minCost(m, n, pen)
    return r, round(time.time() - t0, 2)

rng = random.Random(1)
def rand_pen(m, n, hi=10**5):
    return [[rng.randint(0, hi) for _ in range(n)] for _ in range(m)]

print("316x316 penalty ngẫu nhiên:", timed(316, 316, rand_pen(316, 316)))
print("316x316 penalty = 0       :", timed(316, 316, [[0] * 316 for _ in range(316)]))
print("1 x 100000                :", timed(1, 100000, rand_pen(1, 100000)))
print("100000 x 1                :", timed(100000, 1, rand_pen(100000, 1)))
print("2 x 50000                 :", timed(2, 50000, rand_pen(2, 50000)))

316x316 penalty ngẫu nhiên: (21967145, 0.35)


316x316 penalty = 0       : (15876946, 0.29)
1 x 100000                : (7061280294, 0.19)


100000 x 1                : (7066885698, 0.25)


2 x 50000                 : (2236757502, 0.26)


## 12. Bộ test

Ví dụ đề, edge case, phản ví dụ cho các cách làm sai, đối chiếu ngẫu nhiên: vét cạn ↔ DP theo lớp ↔ Dijkstra ↔ SPFA ↔ code nộp.

In [15]:
S = Solution()
for m, n, pen, expected in EXAMPLES:
    assert brute_force(m, n, pen) == expected if m * n <= 4 else True
    for fn in (layered_dp, dijkstra_solve, spfa_solve):
        assert fn(m, n, pen) == expected, (fn.__name__, m, n)
    assert S.minCost(m, n, pen) == expected
print("Ví dụ đề: mọi cách khớp")

Ví dụ đề: mọi cách khớp


In [16]:
# Edge case
assert S.minCost(1, 2, [[0, 0]]) == 1 + 2              # hành động 1 (lẻ): sang phải, không phạt
assert S.minCost(2, 1, [[0], [0]]) == 1 + 2            # xuống
assert S.minCost(1, 2, [[10**5, 10**5]]) == 3          # đi đúng luật ở bước đầu nên không phạt
assert S.minCost(1, 3, [[0, 5, 9]]) == 6               # chờ 1 lần (phạt 0) rồi đi: 1 + 0 + 2 + 3
assert S.minCost(1, 3, [[7, 0, 0]]) == 1 + 2 + 3 + 0   # đi thẳng, phạt ở (0,1) bằng 0
print("OK")

OK


Với `1 × 3`, `penalty = [[7,0,0]]`: đi thẳng là hành động 1 (lẻ) sang `(0,1)` tốn `2`; hành động 2 (chẵn) sang `(0,2)` là sai luật: `3 + penalty[0][1] = 3 + 0 = 3`. Tổng `1 + 2 + 3 = 6`. Ta thêm vài edge case có phạt lớn để thấy việc "chờ trước" hay "đi sai luật":

In [17]:
assert S.minCost(1, 3, [[0, 9, 0]]) == 6               # chờ ở (0,0) (phạt 0), sang (0,1) (sai luật, phạt 0), sang (0,2) (đúng luật)
assert S.minCost(1, 3, [[9, 0, 0]]) == 1 + 2 + 3 + 0   # đi thẳng: phạt ở (0,1) = 0
assert S.minCost(1, 3, [[9, 9, 0]]) == layered_dp(1, 3, [[9, 9, 0]])
big = [[10**5] * 5 for _ in range(4)]
assert S.minCost(4, 5, big) == layered_dp(4, 5, big)
print("Edge case có phạt lớn: OK")

Edge case có phạt lớn: OK


**Phản ví dụ.** Các lỗi phổ biến (bỏ parity, bỏ "chờ", phạt ở ô đích thay vì ô xuất phát, parity chỉ đổi khi có phạt) đều bị bắt bằng test ngẫu nhiên; ta minh hoạ từng lỗi ở mục 13.

In [18]:
# 1) Vét cạn thật sự trên lưới cực nhỏ (độ sâu đủ lớn, có cắt tỉa)
rng = random.Random(2026)
for it in range(60):
    m, n = rng.choice([(1, 2), (2, 1), (1, 3), (2, 2)])
    pen = [[rng.randint(0, 6) for _ in range(n)] for _ in range(m)]
    ref = brute_force(m, n, pen)
    assert layered_dp(m, n, pen) == ref, (m, n, pen)
    assert dijkstra_solve(m, n, pen) == ref, (m, n, pen)
    assert spfa_solve(m, n, pen) == ref, (m, n, pen)
    assert S.minCost(m, n, pen) == ref, (m, n, pen)
print("60 lưới siêu nhỏ: vét cạn ↔ DP lớp ↔ Dijkstra ↔ SPFA ↔ code nộp")

60 lưới siêu nhỏ: vét cạn ↔ DP lớp ↔ Dijkstra ↔ SPFA ↔ code nộp


In [19]:
# 2) Lưới nhỏ/vừa: DP theo lớp làm trọng tài
rng = random.Random(7)
for it in range(220):
    m = rng.randint(1, 5)
    n = rng.randint(1, 5)
    if m * n < 2:
        continue
    hi = rng.choice([0, 1, 3, 10, 100])
    pen = [[rng.randint(0, hi) for _ in range(n)] for _ in range(m)]
    ref = layered_dp(m, n, pen)
    assert dijkstra_solve(m, n, pen) == ref == spfa_solve(m, n, pen) == S.minCost(m, n, pen), (m, n, pen)
print("220 lưới ≤ 5x5: các cách khớp")

# 3) Lưới lớn: Dijkstra ↔ SPFA ↔ code nộp
rng = random.Random(99)
for it in range(15):
    m, n = rng.choice([(30, 30), (10, 90), (1, 400), (400, 1), (2, 200)])
    pen = [[rng.randint(0, 10**5) for _ in range(n)] for _ in range(m)]
    a = S.minCost(m, n, pen)
    assert a == dijkstra_solve(m, n, pen) == spfa_solve(m, n, pen)
print("15 lưới lớn: Dijkstra ↔ SPFA ↔ code nộp khớp")

220 lưới ≤ 5x5: các cách khớp


15 lưới lớn: Dijkstra ↔ SPFA ↔ code nộp khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Bỏ parity khỏi trạng thái (chỉ lưu `dist[i][j]`)

Nếu mỗi ô chỉ nhớ chi phí nhỏ nhất, ta mất thông tin "hành động kế là lẻ hay chẵn": cùng ô nhưng parity khác nhau thì các bước sau tốn khác nhau, và chi phí nhỏ nhất ở một parity có thể không phải tốt nhất cho tương lai.

In [20]:
def wrong_no_parity(m, n, penalty):
    INF = float("inf")
    dist = [[INF] * n for _ in range(m)]
    dist[0][0] = 1
    pq = [(1, 0, 0, 1)]
    while pq:
        d, i, j, k = heappop(pq)
        if i == m - 1 and j == n - 1:
            return d
        if d > dist[i][j]:
            continue                                  # SAI: bỏ qua trạng thái parity khác dù chi phí cao hơn
        p = penalty[i][j]
        for cost, x, y, nk in state_edges(m, n, penalty, i, j, k):
            if d + cost < dist[x][y]:
                dist[x][y] = d + cost
                heappush(pq, (d + cost, x, y, nk))
    return -1

rng = random.Random(10)
for _ in range(5000):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    if m * n < 2: continue
    pen = [[rng.randint(0, 8) for _ in range(n)] for _ in range(m)]
    if wrong_no_parity(m, n, pen) != S.minCost(m, n, pen):
        print("m,n =", m, n, "penalty =", pen)
        print("sai :", wrong_no_parity(m, n, pen), "| đúng:", S.minCost(m, n, pen))
        break
else:
    raise AssertionError("không tìm thấy phản ví dụ")

m,n = 3 3 penalty = [[3, 5, 8], [7, 6, 7], [1, 5, 8]]
sai : 32 | đúng: 28


**Cách đúng:** `dist[i][j][k]` có chiều parity; hai parity là hai trạng thái riêng.

### Lỗi 2 — Quên hành động **đứng yên**

Ví dụ 2 của đề cần "chờ" (phạt `penalty[0][0] = 0`) để đổi pha. Bỏ nó là bỏ mất lời giải tốt nhất.

In [21]:
def wrong_no_wait(m, n, penalty):
    INF = float("inf")
    dist = [[[INF, INF] for _ in range(n)] for _ in range(m)]
    dist[0][0][1] = 1
    pq = [(1, 0, 0, 1)]
    while pq:
        d, i, j, k = heappop(pq)
        if i == m - 1 and j == n - 1:
            return d
        if d > dist[i][j][k]:
            continue
        for cost, x, y, nk in state_edges(m, n, penalty, i, j, k)[1:]:   # SAI: [1:] bỏ cạnh đứng yên
            if d + cost < dist[x][y][nk]:
                dist[x][y][nk] = d + cost
                heappush(pq, (d + cost, x, y, nk))
    return -1

print("sai :", wrong_no_wait(2, 2, [[0, 7], [3, 2]]))     
print("đúng:", S.minCost(2, 2, [[0, 7], [3, 2]]))          # 7

sai : 10
đúng: 7


**Cách đúng:** luôn thêm cạnh `(i, j, k) → (i, j, k ^ 1)` với chi phí `penalty[i][j]`.

### Lỗi 3 — Cộng `penalty` của ô **đích** thay vì ô **xuất phát**

Đề viết rõ `penalty[i][j]` với `(i, j)` là ô mình **đi từ**. Dùng `penalty[x][y]` cho kết quả khác.

In [22]:
def wrong_penalty_dest(m, n, penalty):
    INF = float("inf")
    dist = [[[INF, INF] for _ in range(n)] for _ in range(m)]
    dist[0][0][1] = 1
    pq = [(1, 0, 0, 1)]
    while pq:
        d, i, j, k = heappop(pq)
        if i == m - 1 and j == n - 1:
            return d
        if d > dist[i][j][k]:
            continue
        p = penalty[i][j]
        if d + p < dist[i][j][k ^ 1]:
            dist[i][j][k ^ 1] = d + p
            heappush(pq, (d + p, i, j, k ^ 1))
        for di, dj in DIRS:
            x, y = i + di, j + dj
            if 0 <= x < m and 0 <= y < n:
                matches = ((di, dj) in ((0, 1), (1, 0))) == (k == 1)
                nd = d + (x + 1) * (y + 1) + (0 if matches else penalty[x][y])   # SAI: ô đích
                if nd < dist[x][y][k ^ 1]:
                    dist[x][y][k ^ 1] = nd
                    heappush(pq, (nd, x, y, k ^ 1))
    return -1

print("sai :", [wrong_penalty_dest(m, n, pen) for m, n, pen, _ in EXAMPLES])
print("đúng:", [e for *_, e in EXAMPLES])

sai : [8, 9, 13]
đúng: [8, 7, 12]


**Cách đúng:** `p = penalty[i][j]` lấy **trước** vòng lặp hướng, dùng cho cả đứng yên lẫn đi sai luật.

### Lỗi 4 — Chỉ đổi parity khi đi sai luật / có phạt

Đề nhấn mạnh: "luật chẵn/lẻ đổi sau **mọi** hành động, bất kể có phạt hay không". Nếu chỉ đổi khi có phạt, ta đã làm sai pha.

In [23]:
def wrong_parity_only_on_penalty(m, n, penalty):
    INF = float("inf")
    dist = [[[INF, INF] for _ in range(n)] for _ in range(m)]
    dist[0][0][1] = 1
    pq = [(1, 0, 0, 1)]
    while pq:
        d, i, j, k = heappop(pq)
        if i == m - 1 and j == n - 1:
            return d
        if d > dist[i][j][k]:
            continue
        p = penalty[i][j]
        if d + p < dist[i][j][k ^ 1]:
            dist[i][j][k ^ 1] = d + p
            heappush(pq, (d + p, i, j, k ^ 1))
        for di, dj in DIRS:
            x, y = i + di, j + dj
            if 0 <= x < m and 0 <= y < n:
                matches = ((di, dj) in ((0, 1), (1, 0))) == (k == 1)
                nk = k if matches else k ^ 1                  # SAI: chỉ đổi parity khi sai luật
                nd = d + (x + 1) * (y + 1) + (0 if matches else p)
                if nd < dist[x][y][nk]:
                    dist[x][y][nk] = nd
                    heappush(pq, (nd, x, y, nk))
    return -1

rng = random.Random(12)
for _ in range(5000):
    m, n = rng.randint(1, 4), rng.randint(2, 4)
    pen = [[rng.randint(0, 8) for _ in range(n)] for _ in range(m)]
    if wrong_parity_only_on_penalty(m, n, pen) != S.minCost(m, n, pen):
        print("m,n =", m, n, "penalty =", pen)
        print("sai :", wrong_parity_only_on_penalty(m, n, pen), "| đúng:", S.minCost(m, n, pen))
        break
else:
    raise AssertionError("không tìm thấy phản ví dụ")

m,n = 4 3 penalty = [[8, 5, 2], [6, 0, 5], [7, 4, 7], [3, 8, 0]]
sai : 30 | đúng: 39


**Cách đúng:** `k ^ 1` sau **mọi** hành động.

## 14. Mẹo & nhận dạng pattern

- **"Luật phụ thuộc số bước (chẵn/lẻ, chia dư `r`)"** → thêm chiều `step mod r` vào trạng thái. Đây là kỹ thuật "đồ thị tầng" (layered graph) / product graph.
- **Chi phí không âm + tìm chi phí nhỏ nhất** → Dijkstra với `heapq`; đặt chi phí ở đầu tuple.
- **Hành động "chờ"** là cạnh tự trỏ (self-loop) biến đổi pha/trạng thái phụ: đừng quên nó.
- **Đừng tách rời** "chi phí vào ô" và "phạt": đọc kỹ ô nào được dùng (ô xuất phát hay ô đích).
- **Lưới lệch** (`1 × 10^5`) vẫn là đồ thị tuyến tính; Dijkstra vẫn chạy tốt, không cần xử lý riêng. Chú ý đệ quy sâu nếu dùng DFS (ở đây ta không dùng).
- Luôn tự viết trọng tài chậm nhưng bám sát chữ đề (DP theo lớp thời gian hoặc vét cạn) để chặn lỗi hiểu đề.

## 15. Biến thể

1. **Không được đứng yên:** bỏ cạnh chờ.
2. **Giới hạn số hành động `T`:** cần DP theo lớp (không dùng Dijkstra thuần).
3. **Luật tuần hoàn `r` thay vì 2:** trạng thái có chiều `step mod r`.
4. **Chi phí vào ô khác** (ví dụ `i + j`, hoặc bảng cho trước): chỉ đổi công thức cạnh.

Ta cài thử biến thể 1 (không đứng yên) và kiểm chứng với DP theo lớp bỏ cạnh đứng yên.

In [24]:
def dijkstra_variant(m, n, penalty, allow_wait=True):
    INF = float("inf")
    dist = [[[INF, INF] for _ in range(n)] for _ in range(m)]
    dist[0][0][1] = 1
    pq = [(1, 0, 0, 1)]
    while pq:
        d, i, j, k = heappop(pq)
        if i == m - 1 and j == n - 1:
            return d
        if d > dist[i][j][k]:
            continue
        for cost, x, y, nk in state_edges(m, n, penalty, i, j, k)[0 if allow_wait else 1:]:
            if d + cost < dist[x][y][nk]:
                dist[x][y][nk] = d + cost
                heappush(pq, (d + cost, x, y, nk))
    return -1

def layered_no_wait(m, n, penalty):
    T = 2 * m * n + 2
    INF = float("inf")
    cur = [[INF] * n for _ in range(m)]
    cur[0][0] = 1
    best = INF
    for t in range(1, T + 1):
        nxt = [[INF] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                if cur[i][j] == INF: continue
                for di, dj in DIRS5[1:]:             # bỏ đứng yên
                    res = action_cost(m, n, penalty, i, j, di, dj, t)
                    if res is None: continue
                    c, (x, y) = res
                    nxt[x][y] = min(nxt[x][y], cur[i][j] + c)
        cur = nxt
        best = min(best, cur[m - 1][n - 1])
    return best

print([dijkstra_variant(m, n, pen, allow_wait=False) for m, n, pen, _ in EXAMPLES])
rng = random.Random(15)
for _ in range(150):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    if m * n < 2: continue
    pen = [[rng.randint(0, 9) for _ in range(n)] for _ in range(m)]
    assert dijkstra_variant(m, n, pen, False) == layered_no_wait(m, n, pen), (m, n, pen)
print("Biến thể không-đứng-yên: khớp DP theo lớp")

[8, 10, 12]
Biến thể không-đứng-yên: khớp DP theo lớp


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi dãy hành động | `O(5^L)` | `O(L)` | trọng tài, lưới 1–4 ô |
| 2. DP theo lớp | `cur[i][j]` sau `t` hành động | `O(N²)` | `O(N)` | trọng tài cho `N ≤ 40`; có giới hạn `T` |
| 3. Dijkstra (nộp) | đồ thị `(i, j, parity)`, heap | `O(N log N)` | `O(N)` | luôn dùng cho đề |
| 4. SPFA | cùng đồ thị, hàng đợi thường | xấu nhất `O(N·E)` | `O(N)` | có cạnh âm; kiểm chứng chéo |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `enter_cost(i, j)` là `(i+1)*(j+1)`. `enter_cost(2, 3)` và tổng `enter_cost` của đường `(0,0)→(0,1)→(0,2)` (kể cả ô đầu) là bao nhiêu?

**Bài 2 (tính tay).** `m=1, n=3`, `penalty=[[0,5,9]]`. Tìm chi phí nhỏ nhất. (Thử cả "đi thẳng" và "chờ ở `(0,0)` trước".)

**Bài 3 (điền TODO).** Hoàn thiện `neighbors(m, n, penalty, i, j, k)` trả về danh sách `(chi_phí, i', j', k')` của mọi hành động từ `(i, j, k)`.

**Bài 4 (tự viết lại).** Viết `my_dijkstra(m, n, penalty)` bằng `heapq` dựa trên `neighbors`.

**Bài 5 (biến thể / mini challenge).** Nếu chỉ được thực hiện **tối đa `T` hành động**, chi phí nhỏ nhất là bao nhiêu (hoặc `-1` nếu không tới được)? Viết `min_cost_within(m, n, penalty, T)` và kiểm chứng bằng vét cạn với độ sâu `T`.

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def neighbors(m, n, penalty, i, j, k):
    out = []
    p = penalty[i][j]
    # TODO 1: cạnh đứng yên -> (p, i, j, k ^ 1)
    # TODO 2: với mỗi hướng (di, dj) trong DIRS, kiểm tra biên
    # TODO 3: khớp luật nếu ((di, dj) in ((0, 1), (1, 0))) == (k == 1); sai luật cộng p
    return out

# Bài 4 — TODO
def my_dijkstra(m, n, penalty):
    # TODO: dist[i][j][k], heap bắt đầu (1, 0, 0, 1), lấy ra đích thì trả chi phí
    pass

# Bài 5 — TODO
def min_cost_within(m, n, penalty, T):
    # TODO: DP theo lớp đúng T hành động (hoặc ít hơn), trả -1 nếu không tới được
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: thay số vào công thức.
- Mức 2: `enter_cost(2,3) = 3·4 = 12`.
- Mức 3: đường `(0,0),(0,1),(0,2)` có phí vào `1, 2, 3` → tổng `6`.

**Bài 2.**
- Mức 1: đi thẳng: hành động 2 (chẵn) sang `(0,2)` là sai luật, phạt `penalty[0][1] = 5`.
- Mức 2: chờ ở `(0,0)` tốn `penalty[0][0] = 0` rồi hành động 2 (chẵn) sang `(0,1)` sai luật: phạt `penalty[0][0] = 0`; hành động 3 (lẻ) sang `(0,2)` đúng luật.
- Mức 3: `1 + 0 + 2 + 0 + 3 = 6`; đi thẳng là `1 + 2 + 3 + 5 = 11`. Đáp án `6`.

**Bài 3.**
- Mức 1: danh sách kết quả bắt đầu với cạnh đứng yên.
- Mức 2: `matches = ((di, dj) in ((0, 1), (1, 0))) == (k == 1)`.
- Mức 3: `out.append(((x + 1) * (y + 1) + (0 if matches else p), x, y, k ^ 1))`.

**Bài 4.**
- Mức 1: `dist = [[[inf, inf] ...]]` và `dist[0][0][1] = 1`.
- Mức 2: bỏ qua bản ghi cũ `d > dist[i][j][k]`.
- Mức 3: lần đầu lấy ra `(m-1, n-1)` thì `return d`.

**Bài 5.**
- Mức 1: giống cách 2 nhưng `T` cho trước.
- Mức 2: `best = min` qua các `t ≤ T`.
- Mức 3: nếu `best` vẫn vô cực thì trả `-1`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
def enter_cost(i, j):
    return (i + 1) * (j + 1)
assert enter_cost(2, 3) == 12
assert sum(enter_cost(0, j) for j in range(3)) == 6

# Bài 2
assert Solution().minCost(1, 3, [[0, 5, 9]]) == 6

# Bài 3
def neighbors_ans(m, n, penalty, i, j, k):
    out = []
    p = penalty[i][j]
    out.append((p, i, j, k ^ 1))
    for di, dj in DIRS:
        x, y = i + di, j + dj
        if 0 <= x < m and 0 <= y < n:
            matches = ((di, dj) in ((0, 1), (1, 0))) == (k == 1)
            out.append(((x + 1) * (y + 1) + (0 if matches else p), x, y, k ^ 1))
    return out

rng = random.Random(3)
for _ in range(100):
    m, n = rng.randint(1, 4), rng.randint(2, 4)
    pen = [[rng.randint(0, 9) for _ in range(n)] for _ in range(m)]
    for i in range(m):
        for j in range(n):
            for k in (0, 1):
                assert neighbors_ans(m, n, pen, i, j, k) == state_edges(m, n, pen, i, j, k)

# Bài 4
def my_dijkstra_ans(m, n, penalty):
    inf = float("inf")
    dist = [[[inf, inf] for _ in range(n)] for _ in range(m)]
    dist[0][0][1] = 1
    pq = [(1, 0, 0, 1)]
    while pq:
        d, i, j, k = heappop(pq)
        if i == m - 1 and j == n - 1:
            return d
        if d > dist[i][j][k]:
            continue
        for cost, x, y, nk in neighbors_ans(m, n, penalty, i, j, k):
            if d + cost < dist[x][y][nk]:
                dist[x][y][nk] = d + cost
                heappush(pq, (d + cost, x, y, nk))
    return -1

for _ in range(200):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    if m * n < 2: continue
    pen = [[rng.randint(0, 20) for _ in range(n)] for _ in range(m)]
    assert my_dijkstra_ans(m, n, pen) == S.minCost(m, n, pen)
print("Bài 1–4 OK")

Bài 1–4 OK


In [27]:
# Bài 5: tối đa T hành động
def min_cost_within_ans(m, n, penalty, T):
    INF = float("inf")
    cur = [[INF] * n for _ in range(m)]
    cur[0][0] = 1
    best = INF
    for t in range(1, T + 1):
        nxt = [[INF] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                if cur[i][j] == INF: continue
                for di, dj in DIRS5:
                    res = action_cost(m, n, penalty, i, j, di, dj, t)
                    if res is None: continue
                    c, (x, y) = res
                    nxt[x][y] = min(nxt[x][y], cur[i][j] + c)
        cur = nxt
        best = min(best, cur[m - 1][n - 1])
    return -1 if best == INF else best

assert min_cost_within_ans(2, 2, [[0, 7], [3, 2]], 3) == 7      # cần đúng 3 hành động
assert min_cost_within_ans(2, 2, [[0, 7], [3, 2]], 2) == 1 + 2 + 4 + 3   # xuống (2) rồi sang phải sai luật (4 + penalty[1][0]=3): 10
assert min_cost_within_ans(2, 2, [[0, 7], [3, 2]], 1) == -1
for _ in range(60):
    m, n = rng.choice([(1, 2), (2, 1), (1, 3), (2, 2)])
    pen = [[rng.randint(0, 6) for _ in range(n)] for _ in range(m)]
    T = rng.randint(1, 5)
    ref = brute_force(m, n, pen, L=T)
    ref = -1 if ref == float("inf") else ref
    assert min_cost_within_ans(m, n, pen, T) == ref, (m, n, pen, T)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đồ thị trạng thái + parity + Dijkstra":**

```python
dist = [[[inf, inf] for _ in range(n)] for _ in range(m)]
dist[0][0][1] = 1                      # đã trả phí vào ô đầu; hành động kế là lẻ
pq = [(1, 0, 0, 1)]
while pq:
    d, i, j, k = heappop(pq)
    if (i, j) == (m - 1, n - 1): return d
    if d > dist[i][j][k]: continue
    p = penalty[i][j]
    relax(i, j, k ^ 1, d + p)           # đứng yên
    for idx, (dx, dy) in enumerate(((-1,0),(0,1),(0,-1),(1,0))):
        x, y = i + dx, j + dy
        if in_grid(x, y):
            relax(x, y, k ^ 1, d + (x+1)*(y+1) + ((idx & 1) ^ k) * p)
```

**Checklist trước khi submit:**

- [ ] Trạng thái có đủ `(i, j, parity)`; parity đổi sau **mọi** hành động.
- [ ] Chi phí khởi đầu là `1` (vào `(0,0)`), không phải `0`.
- [ ] Có cạnh đứng yên với chi phí `penalty[i][j]`.
- [ ] Phạt dùng `penalty` của ô **xuất phát**; chỉ phạt khi hướng không khớp parity.
- [ ] Trả về lần **lấy ra** đầu tiên của đích (không phải lần đẩy vào đầu tiên).
- [ ] Đã đối chiếu với DP theo lớp/vét cạn trên lưới nhỏ.